In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

In [8]:
def singlet():
    q = QuantumCircuit(2)
    q.h(0)
    q.cx(0,1)
    q.x(1)
    q.z(0)
    return q

In [30]:
def rotate_for_measurement(qc, basis, qubit):
    if basis == "Z":
        return
    if basis == "X":
        qc.h(qubit)
        return
    if basis == "W":
        qc.ry(-math.pi/4, qubit)
        return
    if basis == "V":
        qc.ry(math.pi/4, qubit)
        return
    raise Exception("Unknown basis")

In [14]:
def average(c,n): 
    backend = BasicSimulator()
    compiled = transpile(c, backend)
    job_sim = backend.run(compiled, shots=n)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    #print(counts)
    count00 = counts.get("00",0) 
    count01 = counts.get("01",0) 
    count10 = counts.get("10",0) 
    count11 = counts.get("11",0) 
    # The return value includes the conversion from measurement results 0,1 to +1,-1
    # Each 00 means a value of  1 (+1 * +1)
    # Each 01 means a value of -1 (+1 * -1)
    # Each 10 means a value of -1 (-1 * +1)
    # Each 11 means a value of  1 (+1 * +1)
    return (count00 - count01 - count10 + count11) / n 

In [12]:
def circuit_for_AB(alice_basis, bob_basis):
    c = singlet()  
    rotate_for_measurement(c, alice_basis, 0)  
    rotate_for_measurement(c, bob_basis, 1)   
    c.measure_all()
    return c

In [46]:
def randomm():
    theta = math.acos(math.sqrt(1/3))
    q = QuantumCircuit(1)
    q.ry(2*theta, 0)
    q.measure_all()
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    result = backend.run(compiled, shots=1).result()
    counts = result.get_counts(compiled)
    if counts.get("1",0)==1:
        first = 1
    else:
        first = 0
    if first == 0:
        return 0
    q2 = QuantumCircuit(1)
    q2.h(0)
    q2.measure_all()
    compiled2 = transpile(q2, backend)
    result2 = backend.run(compiled2, shots=1).result()
    counts2 = result2.get_counts(compiled2)
    if counts2.get("1",0)==1:
        second = 1
    else:
        second = 0
    if second==0:
        return 1 
    else:
        return 2

In [9]:
q = singlet()
q.measure_all()
backend = BasicSimulator()
compiled = transpile(q, backend)
result = backend.run(compiled, shots=2000).result()
counts = result.get_counts(compiled)
counts

{'10': 999, '01': 1001}

In [11]:
q = QuantumCircuit(1)
q.h(0)  # +
rotate_for_measurement(q, "X", 0)
q.measure_all()
backend = BasicSimulator()
compiled = transpile(q, backend)
counts = backend.run(compiled, shots=2000).result().get_counts(compiled)
counts

{'0': 2000}

In [15]:
Nshots = 5000
XW = average(circuit_for_AB("X","W"), Nshots)
XV = average(circuit_for_AB("X","V"), Nshots)
ZW = average(circuit_for_AB("Z","W"), Nshots)
ZV = average(circuit_for_AB("Z","V"), Nshots)
S = abs(XW - XV + ZW + ZV)
print("XW XV ZW ZV:", XW, XV, ZW, ZV)
print("S =", S)

XW XV ZW ZV: 0.7236 0.7148 -0.6916 0.6904
S = 0.007600000000000051


In [18]:
alice_ops = ["X","W","Z"]  
bob_ops   = ["W","Z","V"] 

In [48]:
from collections import Counter
counter = Counter()
for _ in range(6000):
    counter[randomm()] += 1
print(counter)

Counter({1: 2027, 2: 2020, 0: 1953})


In [42]:
Nkey = 50
shots_per_pair = 1
rounds = (9*Nkey)//2

keyA = []
keyB = []

sums = {("X","W"):0, ("X","V"):0, ("Z","W"):0, ("Z","V"):0}
cnts = {k:0 for k in sums}

for _ in range(rounds):
    ai = randomm()  # 012 X/W/Z
    bj = randomm()  # 012 W/Z/V

    Ab = alice_ops[ai]
    Bb = bob_ops[bj]

    c = singlet()
    rotate_for_measurement(c, Ab, 0)
    rotate_for_measurement(c, Bb, 1)
    c.measure_all()

    backend = BasicSimulator()
    compiled = transpile(c, backend)
    res = backend.run(compiled, shots=1).result()
    counts = res.get_counts(compiled)

    bitstring = next(iter(counts.keys()))  
    a_bit = 1 if bitstring[-1]=='1' else 0
    b_bit = 1 if bitstring[-2]=='1' else 0

    if (ai==1 and bj==0) or (ai==2 and bj==1):
        keyA.append(a_bit)
        keyB.append(1-b_bit) 
        continue

    pair = (Ab, Bb)
    if pair in sums:
        prod = (1 if a_bit==0 else -1) * (1 if b_bit==0 else -1)
        sums[pair] += prod
        cnts[pair] += 1

XW = sums[("X","W")]/cnts[("X","W")] if cnts[("X","W")] else 0
XV = sums[("X","V")]/cnts[("X","V")] if cnts[("X","V")] else 0
ZW = sums[("Z","W")]/cnts[("Z","W")] if cnts[("Z","W")] else 0
ZV = sums[("Z","V")]/cnts[("Z","V")] if cnts[("Z","V")] else 0
S  = abs(XW - XV + ZW + ZV)

ok = all(keyA[i]==keyB[i] for i in range(min(len(keyA),len(keyB))))

print("rounds", rounds)
print("key_len", len(keyA))
print("key_match", ok)
print("S", S)
print("terms", (XW,XV,ZW,ZV))
print("key_sample", keyA[:20])

rounds 225
key_len 46
key_match True
S 2.875175315568023
terms (-0.7419354838709677, 0.6521739130434783, -0.7391304347826086, -0.7419354838709677)
key_sample [1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0]


In [31]:
def circuit_for_AB(alice_basis, bob_basis):
    c = QuantumCircuit(2)
    prepare_singlet(c,0,1)
    rotate_for_measurement(c, alice_basis, 0)
    rotate_for_measurement(c, bob_basis, 1)
    c.measure_all()
    return c

Nshots = 5000
XW = average(circuit_for_AB("X","W"), Nshots)
XV = average(circuit_for_AB("X","V"), Nshots)
ZW = average(circuit_for_AB("Z","W"), Nshots)
ZV = average(circuit_for_AB("Z","V"), Nshots)
S  = abs(XW - XV + ZW + ZV)
print(XW, XV, ZW, ZV, "S=", S)

-0.698 0.7136 -0.6972 -0.7048 S= 2.8136
